# Install Fabric Architecture Review in FOS
Copyright (c) Microsoft Corporation. Licensed under the MIT License.

Run all cells with the existing schema-enabled **FOS_Lakehouse** attached. The installer uses only the packaged runtime in `Files/far/runtime/source`; it creates or updates and publishes **FAR_Runtime** from the packaged requirements before binding the review notebooks. Publication can take several minutes. No application, connection, schedule or access grant is created.

The installer identity needs to manage FAR workspace items, Variable Libraries, directories and schema `far`. Runtime identities need read-only access to `Files/far/runtime` and writes only to their specific output directories/tables, **not** to `Files/far` or its ancestors. Pause running pipelines before upgrading. Existing configuration values are retained.


## Parameters

In [ ]:
PIPELINE_NAME = "Fabric Arch Review Pipeline"
NOTEBOOK_PREFIX = "FabricArchReview"

# --- Governance report (Direct Lake) deployed alongside the pipeline ---
SEMANTIC_MODEL_NAME = "Fabric Arch Review - Governance"
REPORT_NAME = "Fabric Arch Review - Governance"
DATA_AGENT_NAME = "Fabric Arch Review - Data Agent"
DEPLOY_GOLD_REPORT = "true"   # "false" to skip the model + report
DEPLOY_WORKSPACE_OWNER_REPORT = "true"  # no automatic sharing or schedule
OWNER_SEMANTIC_MODEL_NAME = "Fabric Arch Review - Workspace Owner Model"
OWNER_REPORT_NAME = "Fabric Arch Review - Workspace Owner"
OWNER_AGENT_NAME = "Fabric Arch Review - Workspace Owner Agent"
# The data agent is deployed by the separate "05_Agent" notebook (run after the
# pipeline); its pinned SDK is included in FAR_Runtime.
ENDORSE_MODEL = ""                # optional: "Promoted" or "Certified" (blank = skip)
SENSITIVITY_LABEL_ID = ""         # optional: sensitivity label GUID to apply (blank = skip)
ONTOLOGY_NAME = "Fabric_Arch_Review_Estate_Ontology"
DEPLOY_ONTOLOGY = "true"   # "false" to skip the Fabric IQ estate ontology

# Thresholds and the scope/policy declarations, the FUAM/FCA connections and the
# authentication identity (tenant, service principal, Key Vault) are in Variable
# Libraries named <prefix>_Thresholds, <prefix>_Connections and <prefix>_Environment.
# Edit them in Fabric after setup; setup never overwrites them.
FAR_CONFIG_PREFIX = "FAR"


## Verify the runtime, attachment and requested features


In [ ]:
import os, sys, runpy
from pathlib import Path
import notebookutils
ctx = notebookutils.runtime.context
if (ctx.get("defaultLakehouseName") != "FOS_Lakehouse" or not ctx.get("defaultLakehouseId")
        or ctx.get("defaultLakehouseWorkspaceId") != (ctx.get("currentWorkspaceId") or ctx.get("workspaceId"))):
    raise ValueError("Attach this workspace's FOS_Lakehouse as default and restart the session.")
REPO_DIR = "/lakehouse/default/Files/far/runtime/source"
if not Path(REPO_DIR, "fabric", "bootstrap.py").is_file():
    raise FileNotFoundError("FAR runtime not staged. Deploy the architecture-review module before running.")
runpy.run_path(str(Path(REPO_DIR, "fabric", "bootstrap.py")))

from orchestration.fabric_api import FabricClient, definition_part
from orchestration.project import validate_artifact_names
from orchestration.setup import attached_fos_lakehouse, ensure_runtime_environment, load_stage
from orchestration.config_deployment import provision_configuration
from orchestration.folders import find_item, organize_existing
from orchestration.deployment import notebook_parameter

client = FabricClient(lambda: notebookutils.credentials.getToken("pbi"))
wid, lhid = attached_fos_lakehouse(client, ctx)
LAKEHOUSE_NAME = "FOS_Lakehouse"
validate_artifact_names({k: globals()[k] for k in (
    "PIPELINE_NAME", "NOTEBOOK_PREFIX", "SEMANTIC_MODEL_NAME", "REPORT_NAME", "DATA_AGENT_NAME",
    "OWNER_SEMANTIC_MODEL_NAME", "OWNER_REPORT_NAME", "OWNER_AGENT_NAME", "ONTOLOGY_NAME", "FAR_CONFIG_PREFIX",
)})

def enabled(value):
    if str(value).strip().lower() not in ("true", "false"):
        raise ValueError("FAR deployment switches must be true or false.")
    return str(value).strip().lower() == "true"

governance, owner, ontology = map(enabled, (DEPLOY_GOLD_REPORT, DEPLOY_WORKSPACE_OWNER_REPORT, DEPLOY_ONTOLOGY))
if owner and (OWNER_REPORT_NAME == REPORT_NAME or OWNER_SEMANTIC_MODEL_NAME == SEMANTIC_MODEL_NAME
              or OWNER_AGENT_NAME == DATA_AGENT_NAME):
    raise ValueError("Owner items must have names distinct from central items.")
if (ENDORSE_MODEL or SENSITIVITY_LABEL_ID) and not governance:
    raise ValueError("Endorsement and sensitivity labeling require DEPLOY_GOLD_REPORT=true.")
runtime_environment_id = ensure_runtime_environment(client, wid, Path(REPO_DIR) / "requirements.txt")


## Prepare FAR schema, directories and editable configuration


In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
spark.sql("CREATE SCHEMA IF NOT EXISTS far")
files = Path("/lakehouse/default/Files/far")
for relative in ("runs", "orchestration/runs", "_access_grants", "_owner-model-backups", "runtime/bundles"):
    (files / relative).mkdir(parents=True, exist_ok=True)
bindings = provision_configuration(client, wid, FAR_CONFIG_PREFIX)
artifacts = [(runtime_environment_id, "Environment", "FAR/Configuration")]
if ctx.get("currentNotebookId"):
    artifacts.append((ctx["currentNotebookId"], "Notebook", "FAR/Deployment"))
for name in bindings.values():
    artifacts.append((find_item(client, wid, name, "VariableLibrary")["id"], "VariableLibrary", "FAR/Configuration"))

def upsert(name, kind, definition, folder=None):
    item_id = client.upsert_item(wid, name, kind, definition)
    target = folder or ("FAR/Pipelines" if kind == "DataPipeline" else "FAR/Governance")
    artifacts.append((item_id, kind, target))
    return item_id

def stage(suffix, filename, **params):
    folders = {"05_Agent": "FAR/Governance", "06_TargetedReviewSetup": "FAR/Extensions",
               "08_OwnerAgent": "FAR/Workspace Owner"}
    nb = load_stage(Path(REPO_DIR), filename, wid, lhid, bindings, params, environment_id=runtime_environment_id)
    return upsert(NOTEBOOK_PREFIX + "_" + suffix, "Notebook", {
        "format": "ipynb", "parts": [definition_part("notebook-content.ipynb", nb)],
    }, folder=folders.get(suffix, "FAR/Notebooks"))


## Deploy the native review notebooks and four-stage pipeline


In [ ]:
collect_id = stage("01_Collect", "01_collect.ipynb")
analyze_id = stage("02_Analyze", "02_analyze.ipynb")
report_id = stage("03_Report", "03_report.ipynb")
gold_id = stage("04_Gold", "04_gold.ipynb")
agent_id = stage("05_Agent", "05_agent.ipynb", DATA_AGENT_NAME=DATA_AGENT_NAME,
                 SEMANTIC_MODEL_NAME=SEMANTIC_MODEL_NAME, LAKEHOUSE_NAME=LAKEHOUSE_NAME)
# --- Pipeline parameters: only values that change from run to run. Everything else
# (repository, configuration bindings, authentication identity) is stamped at setup or
# read from the Variable Libraries. Override these in the Fabric Run dialog or a trigger. ---
label_keys = {
    "CLIENT_NAME": "Contoso",
    "ENGAGEMENT_NAME": "Fabric Architecture Review",
    "REVIEWER_NAME": "",
}
scope_keys = {
    "WORKSPACE_IDS": "",                  # comma-separated workspace GUIDs (blank = every workspace the identity can see)
}

pipeline_parameters = {k: {"type": "string", "defaultValue": str(v)} for k, v in {**label_keys, **scope_keys}.items()}

common = {k: "@pipeline().parameters." + k for k in label_keys}
common["RUN_ID"] = "@pipeline().RunId"
collect_params = dict(common)
collect_params.update({k: "@pipeline().parameters." + k for k in scope_keys})
analyze_params = {**common, **{k: "@pipeline().parameters." + k for k in scope_keys}}

def _P(d):
    return {k: notebook_parameter(v) for k, v in d.items()}

def activity(name, nbid, params, depends=None):
    return {
        "name": name, "type": "TridentNotebook", "dependsOn": depends or [],
        "policy": {"timeout": "0.12:00:00", "retry": 1, "retryIntervalInSeconds": 120,
                   "secureOutput": False, "secureInput": False},
        "typeProperties": {"notebookId": nbid, "workspaceId": wid, "parameters": _P(params)},
    }

pipe = {"properties": {"parameters": pipeline_parameters, "activities": [
    activity("01 Collect", collect_id, collect_params),
    activity("02 Analyze", analyze_id, analyze_params, [{"activity": "01 Collect", "dependencyConditions": ["Succeeded"]}]),
    activity("03 Report", report_id, common, [{"activity": "02 Analyze", "dependencyConditions": ["Succeeded"]}]),
    activity("04 Gold", gold_id, common, [{"activity": "03 Report", "dependencyConditions": ["Succeeded"]}]),
]}}

pid = upsert(PIPELINE_NAME, "DataPipeline", {"parts": [definition_part("pipeline-content.json", pipe)]})
print("Pipeline:", PIPELINE_NAME, pid)


## 4. Deploy reporting artifacts (governance, optional owner report and ontology)

In [ ]:
# Bootstrap empty gold Delta tables so the Direct Lake model + report deploy
# (and validate column references) even before the pipeline has ever run.
# mode("ignore") is idempotent: it never clobbers tables the pipeline already filled.
from pyspark.sql import SparkSession
from pyspark.sql.types import (StructType, StructField, StringType, LongType,
                               DoubleType, BooleanType, TimestampType)
from reports.powerbi.schema import GOLD_TABLES, key_backfill
from delta.tables import DeltaTable
_spark = SparkSession.builder.getOrCreate()
_T = {"string": StringType(), "int64": LongType(), "double": DoubleType(),
      "boolean": BooleanType(), "dateTime": TimestampType()}
_tbl_root = "abfss://" + wid + "@onelake.dfs.fabric.microsoft.com/" + lhid + "/Tables/far"
print("Bootstrapping empty gold tables (skipped where the pipeline already wrote data)...")
for _t in GOLD_TABLES:
    _sch = StructType([StructField(c.name, _T[c.kind], True) for c in _t.columns])
    _path = _tbl_root + "/" + _t.name
    _spark.createDataFrame([], _sch).write.format("delta").mode("ignore").save(_path)
    # A release can add columns to a table the pipeline already filled. Append no rows with a merged schema so
    # they exist before the model, report and ontology that read them are deployed. Existing rows and columns are untouched.
    _have = {f.name for f in _spark.read.format("delta").load(_path).schema.fields}
    if {c.name for c in _t.columns} - _have:
        _spark.createDataFrame([], _sch).write.format("delta").mode("append").option("mergeSchema", "true").save(_path)
    # A key that a model relationship uses must have no blank, so fill it on rows written before it existed.
    for _key, _cond, _value in key_backfill(_t.name):
        DeltaTable.forPath(_spark, _path).update(condition=_cond, set={_key: _value})
print("  gold tables ready:", ", ".join(t.name for t in GOLD_TABLES))

# Keep one deployment-version row without overwriting Gold run history.
from datetime import datetime, timezone
_far_ver = Path(REPO_DIR, "VERSION").read_text(encoding="utf-8-sig").strip()
_meta_sch = StructType([
    StructField("version", StringType(), True),
    StructField("deployed_at_utc", StringType(), True),
    StructField("repo_url", StringType(), True),
    StructField("branch", StringType(), True),
    StructField("git_ref", StringType(), True),
    StructField("git_sha", StringType(), True),
])
_meta_row = [(_far_ver, datetime.now(timezone.utc).replace(microsecond=0).isoformat(),
              "", "", "vendored", "")]
(_spark.createDataFrame(_meta_row, _meta_sch).write.format("delta")
    .mode("overwrite").option("overwriteSchema", "true").save(_tbl_root + "/meta_deployment"))
print("  meta_deployment recorded: FAR v" + _far_ver + " @ " + "vendored")

if governance or ontology:
    from reports.powerbi.deploy import wait_for_sql_endpoint, model_definition, report_definition
    sql_server, sql_db = wait_for_sql_endpoint(client.get_json, wid, lhid)
if governance:
    mid = upsert(SEMANTIC_MODEL_NAME, "SemanticModel", model_definition(SEMANTIC_MODEL_NAME, sql_server, sql_db))
    rid = upsert(REPORT_NAME, "Report", report_definition(mid))
    import requests
    from reports.powerbi.deploy import refresh_model
    print(refresh_model(requests, {"Authorization": "Bearer " + notebookutils.credentials.getToken("pbi")}, wid, mid))
    if ENDORSE_MODEL:
        from sempy_labs.semantic_model import set_endorsement
        set_endorsement(dataset=SEMANTIC_MODEL_NAME, endorsement=ENDORSE_MODEL, workspace=wid)
    if SENSITIVITY_LABEL_ID:
        from sempy_labs.admin import bulk_set_labels
        bulk_set_labels(item_ids=[mid, rid], label_id=SENSITIVITY_LABEL_ID)
if ontology:
    from reports.ontology.ontology import LakehouseSource, dropped_parts, ontology_definition
    ws_name = client.get_json("/workspaces/" + wid)["displayName"]
    definition = ontology_definition(LakehouseSource(wid, ws_name, lhid, LAKEHOUSE_NAME, sql_server, sql_db), name=ONTOLOGY_NAME)
    oid = upsert(ONTOLOGY_NAME, "Ontology", definition)
    back = client.complete(client.request("POST", f"/workspaces/{wid}/items/{oid}/getDefinition"))
    lost = dropped_parts(definition, back["definition"])
    if lost:
        raise RuntimeError("Ontology readback missing required parts: " + ", ".join(lost))
    print("Ontology deployed. Materialize its graph in Fabric.")

owner_access_notebook_id = ""
if owner:
    from reports.owner.deployment import deploy_owner_reporting
    result = deploy_owner_reporting(
        client, environment_library=bindings["FAR_ENVIRONMENT_LIBRARY"], environment_id=runtime_environment_id,
        repo_dir=Path(REPO_DIR),
        workspace_id=wid, lakehouse_id=lhid, lakehouse_name=LAKEHOUSE_NAME,
        notebook_prefix=NOTEBOOK_PREFIX, model_name=OWNER_SEMANTIC_MODEL_NAME,
        report_name=OWNER_REPORT_NAME, spark=spark,
    )
    owner_access_notebook_id = result["access_notebook_id"]
    artifacts.extend([(owner_access_notebook_id, "Notebook", "FAR/Workspace Owner"),
                      (result["model_id"], "SemanticModel", "FAR/Workspace Owner"),
                      (result["report_id"], "Report", "FAR/Workspace Owner")])
    stage("08_OwnerAgent", "08_owner_agent.ipynb", WORKSPACE_ID=wid, OWNER_SEMANTIC_MODEL_ID=result["model_id"],
          OWNER_AGENT_NAME=OWNER_AGENT_NAME)
stage("06_TargetedReviewSetup", "06_targeted_review_setup.ipynb", WORKSPACE_ID=wid, LAKEHOUSE_ID=lhid,
      CHILD_PIPELINE_ID=pid, PARENT_PIPELINE_NAME=PIPELINE_NAME + " - Targeted Review",
      RUNTIME_ENVIRONMENT_ID=runtime_environment_id,
      OWNER_ACCESS_NOTEBOOK_ID=owner_access_notebook_id)


## Organize artifacts

Keep native FAR item names inside Deployment, Configuration, Pipelines, Notebooks, Governance, Workspace Owner and Extensions. Existing configured optional items are reorganized without being recreated or shared. FOS_Lakehouse and unrelated workspace items are not moved. Folder API errors stop setup; resolve them and rerun.


In [ ]:
configured = [
    (DATA_AGENT_NAME, "DataAgent", "FAR/Governance"),
    (OWNER_AGENT_NAME, "DataAgent", "FAR/Workspace Owner"),
    (OWNER_SEMANTIC_MODEL_NAME, "SemanticModel", "FAR/Workspace Owner"),
    (OWNER_REPORT_NAME, "Report", "FAR/Workspace Owner"),
    (NOTEBOOK_PREFIX + "_07_OwnerAccessSync", "Notebook", "FAR/Workspace Owner"),
    (NOTEBOOK_PREFIX + "_08_OwnerAgent", "Notebook", "FAR/Workspace Owner"),
    (SEMANTIC_MODEL_NAME, "SemanticModel", "FAR/Governance"),
    (REPORT_NAME, "Report", "FAR/Governance"),
    (ONTOLOGY_NAME, "Ontology", "FAR/Governance"),
    (PIPELINE_NAME + " - Targeted Review", "DataPipeline", "FAR/Extensions"),
    (PIPELINE_NAME + " - Targeted Review - Runner", "Notebook", "FAR/Extensions"),
    (PIPELINE_NAME + " - Targeted Review - Completion", "Notebook", "FAR/Extensions"),
]
organize_existing(client, wid, configured, known=artifacts)
print("FAR installed. Review the Thresholds, Connections and Environment Variable Libraries before running", PIPELINE_NAME)
print("Reports: Files/far/runs/<run-id>/report.md. Native model/report:", governance, "| owner reporting:", owner, "| ontology:", ontology)
print("No agent published or app created. To deploy a central agent, enable governance reporting, run Gold and run 05 manually.")


## Optional Workspace Owner report - complete before sharing

Workspace-owner reporting is installed by default. Its RLS does not secure the central report, app or central Agent.

Complete these steps in order; use the [owner reporting guide](https://github.com/microsoft/fabric-architecture-review/blob/main/docs/workspace-owner-report.md) for the full walkthrough and acceptance checklist.

1. **Connect the owner semantic model.** In **Settings -> Gateway and cloud connections**, map it to a shared cloud fixed-identity connection for the FOS_Lakehouse SQL endpoint. The connection identity needs source read access. Keep **Microsoft Entra SSO disabled**, apply the mapping and verify refresh. Reuse the connection on reruns; it is separate from Outlook sign-in.
2. **Populate owner evidence.** Run an owner-enabled, scoped FAR review through Gold. Existing governance-only history is not automatically projected into owner tables.
3. **Synchronize access.** Run **07_OwnerAccessSync** with FOS_Lakehouse attached, or wait for it in an owner-bound targeted parent. Rerun 06 after enabling owner reporting to update the parent. Sync covers all owner-projected workspaces, not only Top N, and does not grant report permissions.
4. **Approve a test reader.** Grant item-scoped **Read** on the owner report/model, with Build and resharing disabled. In the owner model's **Security -> WorkspaceOwner**, add the reader and save.
5. **Validate and schedule.** Test actual read-only users, denied workspaces, expiry, revocation and exports before broader sharing. Schedule 07 **daily**, monitor failures and warnings, and never overlap daily, targeted or manual syncs. Grants expire after **24 hours**. Use the validated owner report URL for notifications.

**Do not grant owner consumers a FAR workspace role, even Viewer, raw Lakehouse/SQL access, Build or Reshare.** Apply organizational labels and sharing restrictions separately; they do not configure RLS. Setup, sync and email never approve reader permissions automatically.

Setup reuses compatible models and upgrades recognized older contracts in place while preserving memberships and connections. Unrecognized changes stop deployment for review. Disabling the deployment flag does not revoke access, stop schedules or delete owner tables.

### Optional Workspace Owner Agent

After Gold and successful 07, run **08_OwnerAgent** separately using its stamped owner model ID and published FAR_Runtime Environment.

The secured owner model is its sole source. Approve **query-only** Agent access, model **Read** and `WorkspaceOwner` membership; Build is not required. Complete actual-reader access tests before sharing. Do not substitute the central Agent or app. See the [owner Agent guide](https://github.com/microsoft/fabric-architecture-review/blob/main/docs/workspace-owner-agent.md).

## 5. Deploy the central data agent (after the pipeline)

1. Complete the FAR pipeline once so Gold tables exist and are populated.
2. Open **`<NOTEBOOK_PREFIX>_05_Agent`**, run it with its published FAR_Runtime Environment.

Verify source availability, publication before use. Keep this Agent restricted to central reviewers; owner-report RLS does not secure it.


## 6. Optional: FUAM-targeted reviews and native Outlook owner email

Both features are optional; the normal FAR pipeline remains available without them.

After a successful scoped review, open **`<NOTEBOOK_PREFIX>_06_TargetedReviewSetup`**. Set `DEPLOY_TARGETED_REVIEW="true"`, configure the FUAM source and review the ranking parameters. Keep the FAR output Lakehouse attached and leave stamped advanced wiring unchanged.

06 creates a **parent pipeline**, **Runner** and **Completion** notebook. Test the parent with an explicit `WORKSPACE_IDS` candidate allow-list and notifications off. Empty selection skips FAR; source failures stop it. The parent does not inherit standalone FAR's saved workspace scope.

For email, follow the [deployment walkthrough](https://github.com/microsoft/fabric-architecture-review/blob/main/fabric/DEPLOYMENT.md): connect and activate the parent's **Office 365 Outlook** activity, set `FAR_REPORT_URL`, then test with `NOTIFICATIONS_ENABLED="true"`. Setup sends nothing. Messages go to eligible direct-user workspace administrators, not automatically to the sender.

Email provides review context and a link, not a task list or access grant. Use the separately validated owner report for workspace owners; a filtered governance link is not an access boundary. Confirm delivery in native monitoring and the mailbox before scheduling. Inspect uncertain outcomes before retrying to avoid duplicate mail.

Schedule the validated parent for recurring targeted reviews, and keep the independent daily 07 access sync when owner reporting is enabled. Setup enables neither schedule.

On redeployment, pause schedules and finish active runs before rerunning 06. It preserves configured email settings but resets the notification default to false and restores generated message bindings. Check schedule overrides before resuming. See the [targeted-review reference](https://github.com/microsoft/fabric-architecture-review/blob/main/docs/targeted-review.md) for parameters and recovery.
